In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!git clone --branch feat/james-ml-pipeline \
    https://github.com/YiMConsulting/YiM-N-ATLAS-Language-Foundry.git \
    /kaggle/working/natlas-foundry

Cloning into '/kaggle/working/natlas-foundry'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 291 (delta 118), reused 246 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (291/291), 164.72 KiB | 4.71 MiB/s, done.
Resolving deltas: 100% (118/118), done.


In [4]:

%cd /kaggle/working/natlas-foundry

!git fetch origin
!git checkout feat/james-ml-pipeline
!git pull --ff-only origin feat/james-ml-pipeline

print("\nCurrent commit:")
!git log -1 --oneline

print("\nWorking tree:")
!git status --short --branch


/kaggle/working/natlas-foundry
Already on 'feat/james-ml-pipeline'
Your branch is up to date with 'origin/feat/james-ml-pipeline'.
From https://github.com/YiMConsulting/YiM-N-ATLAS-Language-Foundry
 * branch            feat/james-ml-pipeline -> FETCH_HEAD
Already up to date.

Current commit:
64e2359 (HEAD -> feat/james-ml-pipeline, origin/feat/james-ml-pipeline) fix:added temp diagnstic to preproces..

Working tree:
## feat/james-ml-pipeline...origin/feat/james-ml-pipeline


In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

print("Hugging face Authentication Successful")


Hugging face Authentication Successful


In [6]:
import subprocess
import inspect

from foundry.training.preprocessing import NAtlasPreprocessor

print("Git commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=False)

print("\nGit status:")
subprocess.run(["git", "status", "--short"], check=False)

print("\nPreprocessor source:")
print(inspect.getsource(NAtlasPreprocessor.preprocess))

Git commit:
64e235931c7e11f45af0d2d8be4cce3706e79f2a

Git status:

Preprocessor source:
    def preprocess(
        self, record: DatasetRecord
    ) -> TokenizedTrainingExample:
        if not record.input.strip():
            raise ValueError("input cannot be empty")
        if not record.target.strip():
            raise ValueError("target cannot be empty")

        messages = [
            {"role": "user", "content": record.input},
            {"role": "assistant", "content": record.target},
        ]

        prompt_result = self.tokenizer.apply_chat_template(
            messages[:1],
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
        )
        full_result = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
        )

        # return_dict=True gives us a mapping containing input_ids.
        prompt_ids = list(prompt_resul

In [8]:
# INTEGRATION TEST..

import torch
from transformers import AutoTokenizer

from foundry.datasets.record import DatasetRecord
from foundry.training.preprocessing import NAtlasPreprocessor
from foundry.training.collator import NAtlasDataCollator


MODEL_ID = "NCAIR1/N-ATLaS"
MAX_LENGTH = 512

# 1. Load the real tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer:", type(tokenizer).__name__)
print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("Padding side:", tokenizer.padding_side)

assert tokenizer.pad_token_id is not None
assert tokenizer.padding_side == "right"

# The previous smoke test confirmed that the existing EOT token
# is available for padding. We should not add a new token.
assert tokenizer.pad_token == "<|eot_id|>"
assert tokenizer.pad_token_id == 128009

# 2. Prepare two representative records.
records = [
    DatasetRecord(
        id="integration-001",
        input="How are you?",
        target="Awa nɔ?",
        language_code="igl",
    ),
    DatasetRecord(
        id="integration-002",
        input="Please tell me how you are feeling today.",
        target="Awa nɔ?",
        language_code="igl",
    ),
]

# 3. Preprocess both records using the real tokenizer.
preprocessor = NAtlasPreprocessor(
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

examples = [preprocessor.preprocess(record) for record in records]

assert len(examples) == 2

for record, example in zip(records, examples):
    assert len(example.input_ids) == len(example.labels)
    assert len(example.input_ids) == len(example.attention_mask)
    assert 0 < len(example.input_ids) <= MAX_LENGTH

    # Each sequence must contain at least one supervised token.
    assert any(label != -100 for label in example.labels)

    # The supervised tokens must be part of the input sequence.
    for token_id, label in zip(example.input_ids, example.labels):
        if label != -100:
            assert token_id == label

    print(
        f"{record.id}: "
        f"tokens={len(example.input_ids)}, "
        f"supervised_tokens={sum(label != -100 for label in example.labels)}"
    )

# 4. Confirm that prompt tokens are masked and the target is supervised.
# The preprocessor is responsible for creating this boundary.
for example in examples:
    first_supervised = next(
        i for i, label in enumerate(example.labels) if label != -100
    )

    assert all(
        label == -100
        for label in example.labels[:first_supervised]
    )

    assert any(
        label != -100
        for label in example.labels[first_supervised:]
    )

# 5. Collate the examples into a training batch.
collator = NAtlasDataCollator(tokenizer=tokenizer)
batch = collator(examples)

assert set(("input_ids", "attention_mask", "labels")).issubset(batch.keys())

input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
labels = batch["labels"]

assert isinstance(input_ids, torch.Tensor)
assert isinstance(attention_mask, torch.Tensor)
assert isinstance(labels, torch.Tensor)

assert input_ids.shape == attention_mask.shape == labels.shape
assert input_ids.ndim == 2
assert input_ids.shape[0] == len(records)
assert input_ids.shape[1] <= MAX_LENGTH

# 6. Verify that padded positions are masked in the labels.
padding_positions = attention_mask == 0

if padding_positions.any():
    assert torch.all(labels[padding_positions] == -100)

    # Padding must not be attended to.
    assert torch.all(attention_mask[padding_positions] == 0)

# 7. Confirm that the batch contains supervised target tokens.
assert torch.any(labels != -100)

print("\nIntegration test passed.")
print("Batch shape:", tuple(input_ids.shape))
print("Input dtype:", input_ids.dtype)
print("Attention-mask dtype:", attention_mask.dtype)
print("Labels dtype:", labels.dtype)


Tokenizer: TokenizersBackend
Pad token: <|eot_id|>
Pad token ID: 128009
EOS token: <|eot_id|>
EOS token ID: 128009
Padding side: right

[DEBUG] Record ID: integration-001
[DEBUG] Tokenizer: TokenizersBackend
[DEBUG] Prompt result type: BatchEncoding
[DEBUG] Full result type: BatchEncoding
[DEBUG] Prompt token count: 39
[DEBUG] Full token count: 46
[DEBUG] Prefix matches: True

[DEBUG] Record ID: integration-002
[DEBUG] Tokenizer: TokenizersBackend
[DEBUG] Prompt result type: BatchEncoding
[DEBUG] Full result type: BatchEncoding
[DEBUG] Prompt token count: 44
[DEBUG] Full token count: 51
[DEBUG] Prefix matches: True
integration-001: tokens=46, supervised_tokens=7
integration-002: tokens=51, supervised_tokens=7

Integration test passed.
Batch shape: (2, 51)
Input dtype: torch.int64
Attention-mask dtype: torch.int64
Labels dtype: torch.int64


In [ ]:
# DEBUGGING SEESION

In [8]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("NCAIR1/N-ATLaS")

records_to_check = [
    {
        "id": "integration-001",
        "input": "How are you?",
        "target": "Awa nɔ?",
    },
    {
        "id": "integration-002",
        "input": "Please tell me how you are feeling today.",
        "target": "Awa nɔ?",
    },
]

for record in records_to_check:
    print("\n" + "=" * 70)
    print("Record:", record["id"])

    prompt_messages = [
        {"role": "user", "content": record["input"]}
    ]
    full_messages = [
        {"role": "user", "content": record["input"]},
        {"role": "assistant", "content": record["target"]},
    ]

    prompt_result = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )
    full_result = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )

    prompt_ids = prompt_result["input_ids"]
    full_ids = full_result["input_ids"]

    common_length = min(len(prompt_ids), len(full_ids))
    first_difference = next(
        (
            i for i in range(common_length)
            if prompt_ids[i] != full_ids[i]
        ),
        None,
    )

    print("Prompt length:", len(prompt_ids))
    print("Full length:", len(full_ids))
    print("First differing token index:", first_difference)
    print("Prompt is prefix:", full_ids[:len(prompt_ids)] == prompt_ids)

    if first_difference is not None:
        start = max(0, first_difference - 8)
        end = first_difference + 12

        print("\nPrompt token IDs:")
        print(prompt_ids[start:end])

        print("\nFull token IDs:")
        print(full_ids[start:end])

        print("\nPrompt decoded segment:")
        print(repr(tokenizer.decode(prompt_ids[start:end])))

        print("\nFull decoded segment:")
        print(repr(tokenizer.decode(full_ids[start:end])))

    print("\nPrompt ending:")
    print(repr(tokenizer.decode(prompt_ids[-15:])))

    print("\nFull sequence around expected boundary:")
    boundary = min(len(prompt_ids), len(full_ids))
    print(repr(tokenizer.decode(
        full_ids[max(0, boundary - 15):boundary + 15]
    )))


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Record: integration-001
Prompt length: 39
Full length: 46
First differing token index: None
Prompt is prefix: True

Prompt ending:
'\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'

Full sequence around expected boundary:
'\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow are you?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAwa nɔ?<|eot_id|>'

Record: integration-002
Prompt length: 44
Full length: 51
First differing token index: None
Prompt is prefix: True

Prompt ending:
'\n\nPlease tell me how you are feeling today.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'

Full sequence around expected boundary:
'\n\nPlease tell me how you are feeling today.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nAwa nɔ?<|eot_id|>'


In [9]:

import inspect
from transformers import AutoTokenizer

from foundry.datasets.record import DatasetRecord
from foundry.training.preprocessing import NAtlasPreprocessor

records = [
    DatasetRecord(
        id="integration-001",
        input="How are you?",
        target="Awa nɔ?",
        language_code="igl",
    ),
    DatasetRecord(
        id="integration-002",
        input="Please tell me how you are feeling today.",
        target="Awa nɔ?",
        language_code="igl",
    ),
]

tokenizer = AutoTokenizer.from_pretrained("NCAIR1/N-ATLaS")
preprocessor = NAtlasPreprocessor(tokenizer=tokenizer, max_length=512)

print("Preprocessing module:", inspect.getfile(NAtlasPreprocessor))
print("\nLoaded preprocess implementation:")
print(inspect.getsource(NAtlasPreprocessor.preprocess))

for record in records:
    print("\n" + "=" * 70)
    print("Record:", record.id)
    print("Input:", repr(record.input))
    print("Target:", repr(record.target))

    messages = [
        {"role": "user", "content": record.input},
        {"role": "assistant", "content": record.target},
    ]

    prompt_result = preprocessor.tokenizer.apply_chat_template(
        messages[:1],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )
    full_result = preprocessor.tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )

    prompt_ids = prompt_result["input_ids"]
    full_ids = full_result["input_ids"]

    print("Prompt result type:", type(prompt_result).__name__)
    print("Prompt IDs type:", type(prompt_ids).__name__)
    print("Full IDs type:", type(full_ids).__name__)
    print("Prompt length:", len(prompt_ids))
    print("Full length:", len(full_ids))
    print("Prefix matches:", full_ids[:len(prompt_ids)] == prompt_ids)

    if full_ids[:len(prompt_ids)] != prompt_ids:
        mismatch = next(
            (
                i for i, (a, b) in enumerate(
                    zip(prompt_ids, full_ids)
                )
                if a != b
            ),
            None,
        )
        print("First mismatch:", mismatch)
        if mismatch is not None:
            print("Prompt tokens:", prompt_ids[max(0, mismatch-5):mismatch+10])
            print("Full tokens:", full_ids[max(0, mismatch-5):mismatch+10])

    try:
        example = preprocessor.preprocess(record)
        print("Preprocessor result: PASS")
        print("Output token count:", len(example.input_ids))
    except Exception as exc:
        print("Preprocessor result:", type(exc).__name__, str(exc))


Preprocessing module: /kaggle/working/natlas-foundry/foundry/training/preprocessing.py

Loaded preprocess implementation:
    def preprocess(
        self, record: DatasetRecord
    ) -> TokenizedTrainingExample:
        if not record.input.strip():
            raise ValueError("input cannot be empty")
        if not record.target.strip():
            raise ValueError("target cannot be empty")

        messages = [
            {"role": "user", "content": record.input},
            {"role": "assistant", "content": record.target},
        ]

        prompt_result = self.tokenizer.apply_chat_template(
            messages[:1],
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
        )
        full_result = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
        )

        prompt_ids = (
            prompt_result["input_ids"]
          

In [14]:
# Inspection.. CHeck whether rewpeated tokenizer calls are consistent..

record = {
    "input": "How are you?",
    "target": "Awa nɔ?",
}

messages = [
    {"role": "user", "content": record["input"]},
    {"role": "assistant", "content": record["target"]},
]

for run in range(1, 5):
    prompt_result = tokenizer.apply_chat_template(
        messages[:1],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )
    full_result = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )

    prompt_ids = list(prompt_result["input_ids"])
    full_ids = list(full_result["input_ids"])

    prefix_matches = full_ids[:len(prompt_ids)] == prompt_ids

    print(
        f"Run {run}: "
        f"prompt={len(prompt_ids)}, "
        f"full={len(full_ids)}, "
        f"prefix_matches={prefix_matches}"
    )

    if not prefix_matches:
        mismatch = next(
            (
                i for i, (a, b) in enumerate(
                    zip(prompt_ids, full_ids)
                )
                if a != b
            ),
            None,
        )

        print("First mismatch:", mismatch)
        if mismatch is not None:
            print("Prompt IDs:", prompt_ids[max(0, mismatch-5):mismatch+10])
            print("Full IDs:", full_ids[max(0, mismatch-5):mismatch+10])

# Now test the preprocessor again using a fresh record.
record_obj = DatasetRecord(
    id="integration-debug",
    input=record["input"],
    target=record["target"],
    language_code="igl",
)

try:
    result = preprocessor.preprocess(record_obj)
    print("\nPreprocessor result: PASS")
    print("Token count:", len(result.input_ids))
except Exception as exc:
    print("\nPreprocessor result:", type(exc).__name__, str(exc))


Run 1: prompt=39, full=46, prefix_matches=True
Run 2: prompt=39, full=46, prefix_matches=True
Run 3: prompt=39, full=46, prefix_matches=True
Run 4: prompt=39, full=46, prefix_matches=True

Preprocessor result: ValueError Prompt tokens are not a prefix of the full conversation. Cannot safely construct assistant-only labels.


In [15]:

class LoggingTokenizer:
    def __init__(self, inner):
        self.inner = inner
        self.call_count = 0

    def apply_chat_template(self, *args, **kwargs):
        self.call_count += 1
        result = self.inner.apply_chat_template(*args, **kwargs)

        ids = result["input_ids"] if isinstance(result, dict) else result

        print(f"\nTokenizer call {self.call_count}")
        print("Messages:", args[0] if args else kwargs.get("conversation"))
        print("Result type:", type(result).__name__)
        print("Input IDs type:", type(ids).__name__)
        print("Token count:", len(ids))
        print("First 12 IDs:", list(ids[:12]))
        print("Last 12 IDs:", list(ids[-12:]))

        return result

    def __getattr__(self, name):
        return getattr(self.inner, name)


# Use a fresh preprocessor and wrap the tokenizer it actually calls.
logging_tokenizer = LoggingTokenizer(tokenizer)
debug_preprocessor = NAtlasPreprocessor(
    tokenizer=logging_tokenizer,
    max_length=512,
)

debug_record = DatasetRecord(
    id="integration-debug",
    input="How are you?",
    target="Awa nɔ?",
    language_code="igl",
)

try:
    debug_example = debug_preprocessor.preprocess(debug_record)
    print("\nPreprocessor result: PASS")
    print("Token count:", len(debug_example.input_ids))
except Exception as exc:
    print("\nPreprocessor result:", type(exc).__name__, str(exc))



Tokenizer call 1
Messages: [{'role': 'user', 'content': 'How are you?'}]
Result type: BatchEncoding
Input IDs type: BatchEncoding
Token count: 2
First 12 IDs: [Encoding(num_tokens=39, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]
Last 12 IDs: [Encoding(num_tokens=39, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

Tokenizer call 2
Messages: [{'role': 'user', 'content': 'How are you?'}, {'role': 'assistant', 'content': 'Awa nɔ?'}]
Result type: BatchEncoding
Input IDs type: BatchEncoding
Token count: 2
First 12 IDs: [Encoding(num_tokens=46, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]
Last 12 IDs: [Encoding(num_tokens=46, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

Preprocessor result: ValueError Prompt tokens are not a prefix of the full conversation. Cannot safely construct assistant-o

In [16]:

result = tokenizer.apply_chat_template(
    [{"role": "user", "content": "How are you?"}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
)

print("Outer type:", type(result))
print("Outer module:", type(result).__module__)
print("Outer keys:", list(result.keys()))
print("Outer repr:", repr(result)[:1200])

value = result["input_ids"]

print("\nValue type:", type(value))
print("Value module:", type(value).__module__)
print("Value repr:", repr(value)[:1200])

print("\nValue attributes:")
for attr in ("keys", "data", "encodings", "input_ids", "ids"):
    try:
        item = getattr(value, attr)
        if callable(item):
            print(attr, "is callable")
        else:
            print(attr, "type:", type(item), "repr:", repr(item)[:500])
    except Exception as exc:
        print(attr, "unavailable:", type(exc).__name__, str(exc))

print("\nOuter encoding information:")
try:
    for index, encoding in enumerate(result.encodings):
        print(
            index,
            "encoding type:", type(encoding),
            "token count:", len(encoding.ids),
            "first IDs:", encoding.ids[:10],
        )
except Exception as exc:
    print("Outer encodings unavailable:", type(exc).__name__, str(exc))


Outer type: <class 'transformers.tokenization_utils_base.BatchEncoding'>
Outer module: transformers.tokenization_utils_base
Outer keys: ['input_ids', 'attention_mask']
Outer repr: {'input_ids': [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 128009, 128006, 882, 128007, 271, 4438, 527, 499, 30, 128009, 128006, 78191, 128007, 271], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Value type: <class 'list'>
Value module: builtins
Value repr: [128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1627, 10263, 220, 2366, 19, 271, 128009, 128006, 882, 128007, 271, 4438, 527, 499, 30, 128009, 128006, 78191, 128007, 271]

Value attributes:
keys unavailable: AttributeError 'list' object has no attribute 'keys'
data unavailable: AttributeError 'list' o